In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [2]:
from typing import Literal

from pydantic import BaseModel


class ContactInfo(BaseModel):
    name: str
    email: str
    plan: Literal["Free", "Pro", "Enterprise"]
    interests: list[str]
    demo_requested: bool


response = client.messages.parse(
    model=model,
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": "Extract: Jane Doe (jane@co.com) wants Enterprise, interested in API and SDKs, wants a demo.",
    }],
    output_format=ContactInfo,
)

contact = response.parsed_output
contact

ContactInfo(name='Jane Doe', email='jane@co.com', plan='Enterprise', interests=['API', 'SDKs'], demo_requested=True)

In [3]:
contact.model_dump()

{'name': 'Jane Doe',
 'email': 'jane@co.com',
 'plan': 'Enterprise',
 'interests': ['API', 'SDKs'],
 'demo_requested': True}

In [4]:
import json

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "plan": {"type": "string", "enum": ["Free", "Pro", "Enterprise"]},
    },
    "required": ["name", "email", "plan"],
    "additionalProperties": False,
}

response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=[{"role": "user", "content": "Extract info: John Smith (john@example.com) wants the Enterprise plan."}],
    output_config={"format": {"type": "json_schema", "schema": schema}},
)

# A cut-off or refused response has no complete JSON to parse.
if response.stop_reason != "end_turn":
    raise RuntimeError(f"Unexpected stop_reason: {response.stop_reason}")

text = next(block.text for block in response.content if block.type == "text")
json.loads(text)

{'name': 'John Smith', 'email': 'john@example.com', 'plan': 'Enterprise'}